<a href="https://colab.research.google.com/github/ParkHangah/AIFFEL_quest_eng/blob/master/LLM_Aplication/LLM03/HuggingFaceProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 한국어 버전 KLUE benchmark 을 이용한 커스텀 프로젝트 직접 만들기


##[출력 보전된 코랩파일 LINK](https://colab.research.google.com/drive/1tdHld4ktzxAp_kikoTSIkbsWjxVCNSSw?usp=sharing)

## [보고서] NSMC 감성 분석 모델 Fine-tuning 결과

#### 1. 실험 개요
- 사용 모델: monologg/koelectra-base-v3-discriminator
- 데이터셋: NSMC (Naver Sentiment Movie Corpus)
- 적용 기술: Batch Size 상향(32), Bucketing(group_by_length), Dynamic Padding


#### 2. 학습결과

##### 2-1. 최초학습 : Epoch 3 / Batch Size: 8

| Epoch | Training Loss | Validation Loss | Accuracy | F1 Score |
| :---: | :---: | :---: | :---: | :---: |
| **1** | 0.2991 | 0.2715 | 0.8999 | 0.8991 |
| **2** | 0.2260 | 0.3428 | 0.9049 | 0.9064 |
| **3** | 0.1683 | 0.4280 | 0.9064 | 0.9069 |

##### 2-2. step4,5적용 (1) : Epoch 3 / Batch Size: 32, Dynamic Padding 적용

| Epoch | Training Loss | Validation Loss | Accuracy | F1 Score |
| :---: | :---: | :---: | :---: | :---: |
| **1** | 0.235924 | 0.248634 | 0.904667 | 0.904901 |
| **2** | 0.201652 | 0.236843 | 0.908167 | 0.908053 |
| **3** | 0.130800 | 0.285101 | 0.908967 | 0.910079 |


#### 📊하드웨어 학습 효율 비교 (Step 5 최적화 효과)

| 지표 | 최초 학습 (Step 3) | Step 4, 5 통합 적용 후 | 개선 정도 |
| :--- | :---: | :---: | :---: |
| **전체 학습 시간 (Total Runtime)** | 3,382s (약 56분) | **2,527s (약 42분)** | **약 25.3% 단축** |
| **초당 처리 샘플 수 (Samples/sec)** | 106.4 | **142.4** | **약 33.8% 향상** |
| **총 학습 스텝 (Global Steps)** | 45,000 steps | **11,250 steps** | **배치 사이즈 확대 반영** |
| **평균 학습 손실 (Train Loss)** | 0.2404 | **0.2038** | **안정성 향상** |

[핵심 분석]
> Bucketing과 Dynamic Padding을 통해 불필요한 패딩(0) 연산을 획기적으로 줄이고 학습 속도와 자원 활용 능력 향상
> 초당 처리량(Throughput)이 33% 이상 증가
> 전체 학습 시간을 14분 절약


#### 📈 모델 성능 및 학습 안정성 비교 (Step 4 최적화 효과)

| 평가 지표 | 최초 학습 (Step 3) | Step 4, 5 통합 적용 후 | 개선 정도 |
| :--- | :---: | :---: | :---: |
| **최고 검증 정확도 (Best Val Acc)** | 0.9064 | **0.9089** | **+0.25%p 향상** |
| **최고 검증 F1 스코어 (Best Val F1)** | 0.9069 | **0.9100** | **+0.31%p 향상** |
| **최종 테스트 정확도 (Test Acc)** | 0.9065 | **0.9065** | **동일 수준 유지** |
| **학습 안정성 (Val Loss 추이)** | 0.27 → 0.42 (상승) | **0.24 → 0.28 (안정)** | **과적합 억제 성공** |

[학습 곡선 및 손실(Loss) 분석]
> 최초 학습: 1 Epoch 이후 Validation Loss가 0.27 → 0.34 → 0.42로 가파르게 상승하며 **과적합(Overfitting)**이 강하게 발생
> 최적화 학습: Validation Loss가 0.24 → 0.23 → 0.28 수준으로 훨씬 낮고 안정적으로 유지
> 해석: 배치 사이즈를 8에서 32로 키운 것이 그래디언트 안정화에 큰 기여를 했으며, 결과적으로 모델이 데이터의 노이즈에 덜 민감해지고 핵심 패턴을 더 잘 학습


#### [종합결론]

- 학습 시간을 25% 단축하면서도 성능 손실이 전혀 없음
- 질적인 성장: 정확도 수치 자체의 향상보다, Validation Loss의 안정화를 통해 모델의 신뢰도가 훨씬 높아짐.
- 한계점: Test 데이터셋 정확도가 90.65%에서 정체된 것으로 보아, 현재의 데이터셋(NSMC)과 모델 체급(koelectra-base)에서는 이 지점이 기술적인 임계치(Ceiling)에 가까워진 것으로 판단